# Nano-vLLM on Google Colab

Run [nano-vllm](https://github.com/GeeeekExplorer/nano-vllm) on a Colab GPU with Qwen3-0.6B.

## GPU requirements

Modern FlashAttention requires **Ampere or newer** (compute capability ≥ 8.0). That means:

| GPU | Compute | Supported? |
|---|---|---|
| T4 (free tier) | 7.5 (Turing) | ❌ no |
| V100 | 7.0 (Volta) | ❌ no |
| L4 (Pro) | 8.9 (Ada) | ✅ yes |
| A100 (Pro) | 8.0 (Ampere) | ✅ yes |
| H100 (Pro+) | 9.0 (Hopper) | ✅ yes |

Set the runtime under *Runtime → Change runtime type* before running cells. The free-tier T4 will fail with `FlashAttention only supports Ampere GPUs or newer.` partway through the example.

**Heads up:** Colab tracks PyTorch's latest release, but `flash-attn` lags by a few minor versions. Step 2 pins torch to a compatible version and requires a one-time runtime restart — after that, the rest runs end-to-end.

## 1. Sanity-check the GPU

If `nvidia-smi` fails, the runtime isn't on a GPU — fix that before continuing.

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv

import subprocess, sys
cap = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"]
).decode().strip().splitlines()[0]
major, minor = (int(x) for x in cap.split("."))
if (major, minor) < (8, 0):
    raise SystemExit(
        f"GPU compute capability {cap} is too old — FlashAttention requires >= 8.0 "
        f"(Ampere or newer). Switch runtime to L4 or A100 (Colab Pro)."
    )
print(f"OK: compute capability {cap} supports FlashAttention.")

## 2. Pin torch to a flash-attn-compatible version (one-time)

This cell is a no-op if you've already restarted with the right torch. If it downgrades torch, **restart the runtime** (*Runtime → Restart session*) and then re-run cells from the top.

If you ever see *"No flash-attn wheel for torch X.Y"* in step 3, bump `TARGET_TORCH` here to a version listed in that error.

In [ ]:
import sys, subprocess, torch

TARGET_TORCH = "2.10"  # newest torch with prebuilt flash-attn 2.x wheels as of 2026-05
current = ".".join(torch.__version__.split(".")[:2])

if current == TARGET_TORCH:
    print(f"torch {torch.__version__} already matches target {TARGET_TORCH}.x — skipping.")
else:
    print(f"torch is {torch.__version__}; pinning to {TARGET_TORCH}.x for flash-attn compatibility...")
    # No --index-url: PyPI's default torch wheel is CUDA-enabled and broadly compatible.
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        f"torch=={TARGET_TORCH}.*", "torchvision", "torchaudio",
    ])
    print("\n*** RESTART THE RUNTIME NOW: Runtime → Restart session ***")
    print("*** Then re-run cells from the top. This cell will become a no-op. ***")

## 3. Install flash-attn (matched wheel)

Queries the [Dao-AILab releases](https://github.com/Dao-AILab/flash-attention/releases) for a prebuilt wheel matching your Python, torch, CUDA, and C++ ABI. Skips the FA4 beta wheels (different package name, incompatible with nano-vllm's imports). Falls back to listing available torch versions if no match is found — bump `TARGET_TORCH` in step 2 and restart if so.

In [ ]:
import sys, re, subprocess, urllib.request, json, torch

py = f"cp{sys.version_info.major}{sys.version_info.minor}"
torch_ver = ".".join(torch.__version__.split(".")[:2])
cuda_major = torch.version.cuda.split(".")[0]
abi = "TRUE" if torch._C._GLIBCXX_USE_CXX11_ABI else "FALSE"
print(f"env: python={py} torch={torch_ver} cuda={cuda_major} abi={abi}")

with urllib.request.urlopen(
    "https://api.github.com/repos/Dao-AILab/flash-attention/releases?per_page=50"
) as r:
    releases = json.load(r)

wheel_pat = re.compile(
    rf"^flash_attn-([\d.]+(?:\.post\d+)?)\+cu(\d+)torch([\d.]+)cxx11abi(TRUE|FALSE)-{py}-{py}-linux_x86_64\.whl$"
)

candidates = []  # (torch_ver, url, fa_ver)
for rel in releases:
    for asset in rel.get("assets", []):
        m = wheel_pat.match(asset["name"])
        if not m:
            continue
        fa_ver, cu, t_ver, w_abi = m.groups()
        if cu.startswith(cuda_major) and w_abi == abi:
            candidates.append((t_ver, asset["browser_download_url"], fa_ver))

if not candidates:
    raise RuntimeError(
        f"No flash-attn 2.x/3.x wheels found at all for python={py} cuda={cuda_major} abi={abi}."
    )

exact = [c for c in candidates if c[0] == torch_ver]
if exact:
    url = exact[0][1]
    print(f"Installing: {url.rsplit('/', 1)[-1]}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", url])
else:
    avail = sorted({c[0] for c in candidates}, key=lambda v: tuple(map(int, v.split("."))))
    raise RuntimeError(
        f"No flash-attn wheel for torch {torch_ver}. Available torch versions: {avail}. "
        f"Update TARGET_TORCH in step 2 to one of these and restart the runtime."
    )

## 4. Install nano-vllm and its other deps

In [ ]:
!pip install -q "triton>=3.0.0" "transformers>=4.51.0" xxhash "huggingface_hub[cli]"
!pip install -q git+https://github.com/GeeeekExplorer/nano-vllm.git

## 5. Download Qwen3-0.6B

~1.2 GB. Goes to `/content/Qwen3-0.6B/`.

In [ ]:
!hf download Qwen/Qwen3-0.6B --local-dir /content/Qwen3-0.6B

## 6. Run the example

Mirrors `example.py`. `enforce_eager=True` skips CUDA-graph capture so startup is fast.

In [ ]:
from nanovllm import LLM, SamplingParams
from transformers import AutoTokenizer

MODEL = "/content/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
llm = LLM(MODEL, enforce_eager=True, tensor_parallel_size=1)

prompts = [
    "Introduce yourself in one sentence.",
    "List the first ten prime numbers.",
]
chat_prompts = [
    tokenizer.apply_chat_template(
        [{"role": "user", "content": p}],
        tokenize=False,
        add_generation_prompt=True,
    )
    for p in prompts
]

sampling_params = SamplingParams(temperature=0.6, max_tokens=256)
outputs = llm.generate(chat_prompts, sampling_params)

for p, out in zip(prompts, outputs):
    print(f"\n>>> {p}\n{out['text']}")

## 7. Throughput benchmark (optional)

Scaled-down version of `bench.py` — 32 random sequences instead of 256 so it fits a free-tier L4 / A100 comfortably. Bump `num_seqs` for bigger GPUs.

Reuses the `llm` from step 6 (which has `enforce_eager=True`). This means no CUDA graph capture, so these numbers will be lower than the upstream README's headline throughput. For CUDA-graph numbers, restart the runtime, skip step 6, and create a fresh `llm` here with `enforce_eager=False`.

In [ ]:
import time
from random import randint, seed
from nanovllm import SamplingParams

seed(0)
num_seqs = 32
prompt_token_ids = [
    [randint(0, 10000) for _ in range(randint(100, 512))]
    for _ in range(num_seqs)
]
sampling_params = [
    SamplingParams(temperature=0.6, ignore_eos=True, max_tokens=randint(100, 512))
    for _ in range(num_seqs)
]

llm.generate(["warmup"], SamplingParams(max_tokens=8), use_tqdm=False)

t = time.time()
llm.generate(prompt_token_ids, sampling_params, use_tqdm=False)
elapsed = time.time() - t

total = sum(sp.max_tokens for sp in sampling_params)
print(f"{num_seqs} seqs, {total} tokens, {elapsed:.2f}s, {total/elapsed:.1f} tok/s")